In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q sentence-transformers FlagEmbedding rank_bm25 pytrec_eval datasets transformers accelerate
print("✅ Packages installed")


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.6 MB/s eta 0:00:00
✅ Packages installed


In [ ]:
import os, json, ast, gc, warnings, pickle, random
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss
from transformers import EarlyStoppingCallback
from rank_bm25 import BM25Okapi

warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()
random.seed(42)
np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
start_path     = '/content/drive/MyDrive/TaskB/TaskB'
STAGE1_PATH    = os.path.join(start_path, 'model_stage1')
STAGE2_PATH    = os.path.join(start_path, 'model_stage2')
STAGE3_PATH    = os.path.join(start_path, 'model_stage3_hardneg')
EMB_CACHE      = os.path.join(start_path, 'corpus_emb_cache.pkl')
PRED_PATH      = os.path.join(start_path, 'predictions.trec')

# ── Model config (T4-safe) ───────────────────────────────────────────────────
BASE_MODEL     = 'BAAI/bge-base-en-v1.5'     # 109M — best retrieval/size tradeoff
RERANK_MODEL   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'  # 22M — tiny + strong
BGE_Q_PREFIX   = "Represent this sentence for searching relevant passages: "

# ── Hyperparams ──────────────────────────────────────────────────────────────
MAX_TERMS      = 120     # was 80 — free gain from more context
TRAIN_BS       = 32      # safe for bge-base on T4 with fp16
EVAL_BS        = 64
RETRIEVAL_K    = 100     # bi-encoder retrieves top-100
RERANK_K       = 50      # reranker re-scores top-50
PRF_K          = 3       # PRF: expand with top-3 skill texts
HARD_NEG_K     = 5       # hard negatives per anchor to mine

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Config ready | Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


✅ Config ready | Device: cuda
   GPU : Tesla T4
   VRAM: 15.6 GB


/tmp/ipykernel_1485/3155325299.py:8: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
/tmp/ipykernel_1485/3155325299.py:9: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments
/tmp/ipykernel_1485/3155325299.py:11: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss


In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
with open(os.path.join(start_path, 'training', 'jobid2terms.json'), 'r', encoding='utf-8') as f:
    job_dict = json.load(f)
with open(os.path.join(start_path, 'training', 'skillid2terms.json'), 'r', encoding='utf-8') as f:
    skill_dict = json.load(f)

job2skill_df = pd.read_csv(
    os.path.join(start_path, 'training', 'job2skill.tsv'),
    sep='\t', header=None, names=['job_id', 'skill_id', 'essential']
).dropna()

queries_df = pd.read_csv(os.path.join(start_path, 'validation', 'queries'), sep='\t').dropna()
corpus_df  = pd.read_csv(os.path.join(start_path, 'validation', 'corpus_elements'), sep='\t').dropna(subset=['c_id'])
qrels_df   = pd.read_csv(
    os.path.join(start_path, 'validation', 'qrels.tsv'),
    sep='\t', header=None, names=['q_id', 'dummy', 'c_id', 'relevance']
).dropna()

# ── Helper text builders ─────────────────────────────────────────────────────
def get_job_text(job_id):
    t = " ".join(job_dict.get(str(job_id), []))
    return " ".join(t.split()[:MAX_TERMS]) if t.strip() else None

def get_skill_text(skill_id):
    t = " ".join(skill_dict.get(str(skill_id), []))
    return " ".join(t.split()[:MAX_TERMS]) if t.strip() else None

# ── Validation corpus + queries ──────────────────────────────────────────────
queries_val = {
    str(row['q_id']): " ".join(str(row['jobtitle']).split()[:MAX_TERMS])
    if pd.notna(row['jobtitle']) else "unknown"
    for _, row in queries_df.iterrows()
}

corpus_val = {}
for _, row in corpus_df.iterrows():
    cid = str(row['c_id'])
    try:
        aliases = ast.literal_eval(row['skill_aliases'])
        text = " ".join(" ".join(aliases).split()[:MAX_TERMS])
    except:
        text = " ".join(str(row['skill_aliases']).split()[:MAX_TERMS])
    corpus_val[cid] = text if text.strip() else "unknown"

queries_eval  = {}
relevant_docs = {}
for qid, qtext in queries_val.items():
    pos = qrels_df[(qrels_df['q_id'].astype(str)==qid)&(qrels_df['relevance']>0)]['c_id'].astype(str).tolist()
    if pos:
        queries_eval[qid]  = qtext
        relevant_docs[qid] = set(pos)

# ── Build job→skills map (all skills per job, with essential flag) ────────────
job2all_skills = {}   # {job_id: [(skill_id, essential_bool), ...]}
for _, row in job2skill_df.iterrows():
    jid = str(row['job_id'])
    sid = str(row['skill_id'])
    ess = str(row['essential']).strip().lower() == 'essential'
    job2all_skills.setdefault(jid, []).append((sid, ess))

print(f"✅ Data loaded")
print(f"   Jobs in training  : {len(job2all_skills)}")
print(f"   Val queries       : {len(queries_eval)}")
print(f"   Corpus skills     : {len(corpus_val)}")


✅ Data loaded
   Jobs in training  : 3011
   Val queries       : 304
   Corpus skills     : 9052


In [ ]:
# ── Stage 1 dataset: all job→skill pairs ─────────────────────────────────────
anchors_all, positives_all = [], []

for jid, skill_list in job2all_skills.items():
    job_text = get_job_text(jid)
    if not job_text:
        continue
    for sid, ess in skill_list:
        skill_text = get_skill_text(sid)
        if skill_text:
            anchors_all.append(job_text)
            positives_all.append(skill_text)
from transformers.trainer_utils import get_last_checkpoint


train_ds_all = Dataset.from_dict({"anchor": anchors_all, "positive": positives_all})
print(f"✅ Stage 1 dataset: {len(train_ds_all)} pairs")


✅ Stage 1 dataset: 114699 pairs


In [ ]:
# ── Stage 2 dataset: essential pairs only (curriculum sharpening) ─────────────
anchors_ess, positives_ess = [], []

for jid, skill_list in job2all_skills.items():
    job_text = get_job_text(jid)
    if not job_text:
        continue
    for sid, ess in skill_list:
        if not ess:
            continue
        skill_text = get_skill_text(sid)
        if skill_text:
            anchors_ess.append(job_text)
            positives_ess.append(skill_text)

train_ds_ess = Dataset.from_dict({"anchor": anchors_ess, "positive": positives_ess})
print(f"✅ Stage 2 dataset: {len(train_ds_ess)} essential pairs")


✅ Stage 2 dataset: 62480 essential pairs


In [ ]:
# ── STAGE 1: Train on ALL pairs (broad contrastive learning) ─────────────────
evaluator  = InformationRetrievalEvaluator(queries_eval, corpus_val, relevant_docs, name='val_ir')
model      = SentenceTransformer(BASE_MODEL)
loss_fn    = MultipleNegativesRankingLoss(model)
eval_steps = max(1, len(train_ds_all) // (TRAIN_BS * 10))   # ~10 saves per epoch

args1 = SentenceTransformerTrainingArguments(
    output_dir=STAGE1_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=TRAIN_BS,
    eval_strategy="steps", eval_steps=eval_steps,
    save_strategy="steps", save_steps=eval_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ir_cosine_map@100",
    warmup_ratio=0.1,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=eval_steps,
)

trainer1 = SentenceTransformerTrainer(
    model=model, args=args1,
    train_dataset=train_ds_all,
    evaluator=evaluator, loss=loss_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("🚀 Stage 1: Training on ALL pairs...")
trainer1.train(resume_from_checkpoint=None)
model.save_pretrained(STAGE1_PATH)
print(f"✅ Stage 1 done → {STAGE1_PATH}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Exception ignored in: <function tqdm.__del__ at 0x7e13e5d167a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1147, in __del__
    def __del__(self):

KeyboardInterrupt: 


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
# ── STAGE 2: Fine-tune on ESSENTIAL pairs (curriculum sharpening) ─────────────
from sentence_transformers.evaluation import InformationRetrievalEvaluator
evaluator  = InformationRetrievalEvaluator(queries_eval, corpus_val, relevant_docs, name='val_ir')
import glob

_ckpts = sorted(glob.glob(f"{STAGE1_PATH}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
model = SentenceTransformer(STAGE1_PATH if os.path.exists(f"{STAGE1_PATH}/config.json") else _ckpts[-1])

loss_fn    = MultipleNegativesRankingLoss(model)
eval_steps = max(1, len(train_ds_ess) // (TRAIN_BS * 10))

args2 = SentenceTransformerTrainingArguments(
    output_dir=STAGE2_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=TRAIN_BS,
    eval_strategy="steps", eval_steps=eval_steps,
    save_strategy="steps", save_steps=eval_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ir_cosine_map@100",
    warmup_ratio=0.05,
    learning_rate=1e-5,             # lower LR for fine-grained stage
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=eval_steps,
)

trainer2 = SentenceTransformerTrainer(
    model=model, args=args2,
    train_dataset=train_ds_ess,
    evaluator=evaluator, loss=loss_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("🚀 Stage 2: Training on ESSENTIAL pairs only...")
trainer2.train(resume_from_checkpoint=None)
model.save_pretrained(STAGE2_PATH)
print(f"✅ Stage 2 done → {STAGE2_PATH}")


Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

🚀 Stage 2: Training on ESSENTIAL pairs only...
{'loss': '1.26', 'grad_norm': '9.706', 'learning_rate': '6.621e-06', 'epoch': '0.09985'}
{'eval_val_ir_cosine_accuracy@1': '0.523', 'eval_val_ir_cosine_accuracy@3': '0.8059', 'eval_val_ir_cosine_accuracy@5': '0.898', 'eval_val_ir_cosine_accuracy@10': '0.9638', 'eval_val_ir_cosine_precision@1': '0.523', 'eval_val_ir_cosine_precision@3': '0.4715', 'eval_val_ir_cosine_precision@5': '0.4789', 'eval_val_ir_cosine_precision@10': '0.4688', 'eval_val_ir_cosine_recall@1': '0.003214', 'eval_val_ir_cosine_recall@3': '0.008386', 'eval_val_ir_cosine_recall@5': '0.01406', 'eval_val_ir_cosine_recall@10': '0.02771', 'eval_val_ir_cosine_ndcg@10': '0.4755', 'eval_val_ir_cosine_mrr@10': '0.6811', 'eval_val_ir_cosine_map@100': '0.1746', 'eval_runtime': '7.346', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '0.09985'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.154', 'grad_norm': '8.033', 'learning_rate': '9.828e-06', 'epoch': '0.1997'}
{'eval_val_ir_cosine_accuracy@1': '0.5428', 'eval_val_ir_cosine_accuracy@3': '0.8322', 'eval_val_ir_cosine_accuracy@5': '0.9046', 'eval_val_ir_cosine_accuracy@10': '0.9638', 'eval_val_ir_cosine_precision@1': '0.5428', 'eval_val_ir_cosine_precision@3': '0.4879', 'eval_val_ir_cosine_precision@5': '0.4717', 'eval_val_ir_cosine_precision@10': '0.477', 'eval_val_ir_cosine_recall@1': '0.003294', 'eval_val_ir_cosine_recall@3': '0.008627', 'eval_val_ir_cosine_recall@5': '0.01401', 'eval_val_ir_cosine_recall@10': '0.02812', 'eval_val_ir_cosine_ndcg@10': '0.4833', 'eval_val_ir_cosine_mrr@10': '0.6922', 'eval_val_ir_cosine_map@100': '0.1771', 'eval_runtime': '7.78', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '0.1997'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'train_runtime': '1388', 'train_samples_per_second': '135.1', 'train_steps_per_second': '4.222', 'train_loss': '1.002', 'epoch': '1.498'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Stage 2 done → /content/drive/MyDrive/TaskB/TaskB/model_stage2


In [ ]:
# ── HARD NEGATIVE MINING ──────────────────────────────────────────────────────
print("Loading Stage 2 model for mining...")
miner_model = SentenceTransformer(STAGE2_PATH)
miner_model.eval()

# ── Encode ALL skills in corpus ──────────────────────────────────────────────
all_skill_ids   = list(skill_dict.keys())
all_skill_texts = [get_skill_text(sid) for sid in all_skill_ids]
valid_mask      = [t is not None for t in all_skill_texts]
all_skill_ids   = [s for s, v in zip(all_skill_ids, valid_mask) if v]
all_skill_texts = [t for t, v in zip(all_skill_texts, valid_mask) if v]

print(f"Encoding {len(all_skill_texts)} skills for mining...")
skill_embeddings = miner_model.encode(
    all_skill_texts, batch_size=EVAL_BS,
    convert_to_tensor=True, normalize_embeddings=True,
    show_progress_bar=True
)

# ── Mine hard negatives ───────────────────────────────────────────────────────
anchors_hn, positives_hn, negatives_hn = [], [], []
unique_jobs = list(job2all_skills.keys())

print(f"Mining hard negatives for {len(unique_jobs)} jobs...")
for i, jid in enumerate(unique_jobs):
    if (i+1) % 500 == 0:
        print(f"  {i+1}/{len(unique_jobs)} jobs processed...")

    job_text = get_job_text(jid)
    if not job_text:
        continue

    true_sids = set(sid for sid, _ in job2all_skills[jid])

    # encode query
    q_emb = miner_model.encode(
        BGE_Q_PREFIX + job_text,
        convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    sims = util.dot_score(q_emb, skill_embeddings)[0].cpu().numpy()

    # rank all skills, exclude true positives
    # skip top-20 (too ambiguous), take next HARD_NEG_K (clearly wrong but relevant)
    ranked_idxs    = np.argsort(sims)[::-1]
    candidate_idxs = [idx for idx in ranked_idxs if all_skill_ids[idx] not in true_sids]
    hard_neg_idxs  = candidate_idxs[20:20 + HARD_NEG_K]

    if not hard_neg_idxs:
        continue

    # only essential positives — clearest training signal
    for sid, ess in job2all_skills[jid]:
        if not ess:
            continue
        pos_text = get_skill_text(sid)
        if not pos_text:
            continue
        for neg_idx in hard_neg_idxs:
            anchors_hn.append(job_text)
            positives_hn.append(pos_text)
            negatives_hn.append(all_skill_texts[neg_idx])

print(f"✅ Mined {len(anchors_hn)} hard negative triplets")

# free miner model memory
del miner_model, skill_embeddings
gc.collect()
torch.cuda.empty_cache()

# save to disk
hn_cache = os.path.join(start_path, 'hard_negatives_cache.pkl')
with open(hn_cache, 'wb') as f:
    pickle.dump({'anchors': anchors_hn, 'positives': positives_hn, 'negatives': negatives_hn}, f)
print(f"💾 Hard negatives cached → {hn_cache}")

Loading Stage 2 model for mining...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 13939 skills for mining...


Batches:   0%|          | 0/218 [00:00<?, ?it/s]

Mining hard negatives for 3011 jobs...
  500/3011 jobs processed...
  1000/3011 jobs processed...
  1500/3011 jobs processed...
  2000/3011 jobs processed...
  2500/3011 jobs processed...
  3000/3011 jobs processed...
✅ Mined 312400 hard negative triplets
💾 Hard negatives cached → /content/drive/MyDrive/TaskB/TaskB/hard_negatives_cache.pkl


In [ ]:
# ── (RECOVERY) Load hard negatives from cache if you crashed ─────────────────
# Skip this cell if you just ran the mining cell above
# Uncomment below if you need to reload:

hn_cache = os.path.join(start_path, 'hard_negatives_cache.pkl')
with open(hn_cache, 'rb') as f:
    hn_data = pickle.load(f)
anchors_hn   = hn_data['anchors']
positives_hn = hn_data['positives']
negatives_hn = hn_data['negatives']
print(f"✅ Loaded {len(anchors_hn)} hard negative triplets from cache")
print("(Skipped — cache recovery not needed)")


✅ Loaded 573495 hard negative triplets from cache
(Skipped — cache recovery not needed)


In [ ]:
# ── STAGE 3: Retrain with HARD NEGATIVES (MNRL) ───────────────────────────────
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from transformers import EarlyStoppingCallback
from datasets import Dataset
import glob, os, random

# Shuffle before subsampling for diverse coverage
MAX_TRIPLETS = 50_000
indices = list(range(len(anchors_hn)))
random.shuffle(indices)
indices = indices[:MAX_TRIPLETS]

anchors_s   = [anchors_hn[i]   for i in indices]
positives_s = [positives_hn[i] for i in indices]
negatives_s = [negatives_hn[i] for i in indices]

train_ds_hn = Dataset.from_dict({
    "anchor":   anchors_s,
    "positive": positives_s,
    "negative": negatives_s,
})
print(f"✅ Triplet dataset: {len(train_ds_hn)} samples (shuffled)")

evaluator = InformationRetrievalEvaluator(queries_eval, corpus_val, relevant_docs, name='val_ir')

# Load Stage 2 model
_s2_ckpts = sorted(glob.glob(os.path.join(STAGE2_PATH, "checkpoint-*")),
                   key=lambda p: int(p.split("-")[-1]))
if os.path.exists(os.path.join(STAGE2_PATH, "config.json")):
    model = SentenceTransformer(STAGE2_PATH)
    print(f"✅ Loaded Stage 2 model from {STAGE2_PATH}")
elif _s2_ckpts:
    model = SentenceTransformer(_s2_ckpts[-1])
    print(f"⚠️  Loaded from last checkpoint: {_s2_ckpts[-1]}")
else:
    raise FileNotFoundError(f"No Stage 2 model found in {STAGE2_PATH}.")

loss_fn_hn = MultipleNegativesRankingLoss(model)

total_steps  = len(train_ds_hn) // TRAIN_BS
eval_steps   = max(1, total_steps // 5)       # 5 evals per epoch
warmup_steps = int(0.1 * total_steps)

args3 = SentenceTransformerTrainingArguments(
    output_dir=STAGE3_PATH,
    num_train_epochs=1,
    per_device_train_batch_size=TRAIN_BS,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_ir_cosine_map@100",
    warmup_steps=warmup_steps,
    learning_rate=2e-6,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,
    logging_steps=eval_steps,
)

trainer3 = SentenceTransformerTrainer(
    model=model, args=args3,
    train_dataset=train_ds_hn,
    evaluator=evaluator, loss=loss_fn_hn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("🚀 Stage 3: Training with HARD NEGATIVES (MNRL)...")
trainer3.train(resume_from_checkpoint=None)
model.save_pretrained(STAGE3_PATH)
print(f"✅ Stage 3 done → {STAGE3_PATH}")

✅ Triplet dataset: 50000 samples (shuffled)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Loaded Stage 2 model from /content/drive/MyDrive/TaskB/TaskB/model_stage2


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

🚀 Stage 3: Training with HARD NEGATIVES (MNRL)...
{'loss': '1.665', 'grad_norm': '10.73', 'learning_rate': '1.78e-06', 'epoch': '0.1996'}
{'eval_val_ir_cosine_accuracy@1': '0.5592', 'eval_val_ir_cosine_accuracy@3': '0.8355', 'eval_val_ir_cosine_accuracy@5': '0.9243', 'eval_val_ir_cosine_accuracy@10': '0.9671', 'eval_val_ir_cosine_precision@1': '0.5592', 'eval_val_ir_cosine_precision@3': '0.4967', 'eval_val_ir_cosine_precision@5': '0.4908', 'eval_val_ir_cosine_precision@10': '0.4681', 'eval_val_ir_cosine_recall@1': '0.003357', 'eval_val_ir_cosine_recall@3': '0.00886', 'eval_val_ir_cosine_recall@5': '0.0147', 'eval_val_ir_cosine_recall@10': '0.0279', 'eval_val_ir_cosine_ndcg@10': '0.4818', 'eval_val_ir_cosine_mrr@10': '0.7047', 'eval_val_ir_cosine_map@100': '0.176', 'eval_runtime': '7.127', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '0.1996'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'train_runtime': '667.4', 'train_samples_per_second': '74.91', 'train_steps_per_second': '2.342', 'train_loss': '1.575', 'epoch': '0.7985'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Stage 3 done → /content/drive/MyDrive/TaskB/TaskB/model_stage3_hardneg


In [ ]:
# ── Encode corpus + build BM25 index ─────────────────────────────────────────
best_model = SentenceTransformer(STAGE3_PATH)
best_model.eval()

cids         = list(corpus_val.keys())
corpus_texts = [corpus_val[cid] for cid in cids]

print(f"Encoding {len(corpus_texts)} corpus skills...")
corpus_embeddings = best_model.encode(
    corpus_texts, batch_size=EVAL_BS,
    convert_to_tensor=True, normalize_embeddings=True,
    show_progress_bar=True
)

# BM25 index (CPU, zero VRAM)
print("Building BM25 index...")
tokenized_corpus = [t.lower().split() for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

# Save to Drive — so crash won't force re-encoding
with open(EMB_CACHE, 'wb') as f:
    pickle.dump({'cids': cids, 'embeddings': corpus_embeddings.cpu().numpy()}, f)
print(f"✅ Corpus encoded | BM25 ready | Cache → {EMB_CACHE}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 9052 corpus skills...


Batches:   0%|          | 0/142 [00:00<?, ?it/s]

Building BM25 index...
✅ Corpus encoded | BM25 ready | Cache → /content/drive/MyDrive/TaskB/TaskB/corpus_emb_cache.pkl


In [ ]:
# ── (RECOVERY) Reload corpus embeddings from cache if crashed ────────────────
# Run this cell instead of the encoding cell above if you crashed

with open(EMB_CACHE, 'rb') as f:
    cache = pickle.load(f)
cids             = cache['cids']
corpus_texts     = [corpus_val[cid] for cid in cids]
corpus_embeddings= torch.tensor(cache['embeddings']).to(device)
tokenized_corpus = [t.lower().split() for t in corpus_texts]
bm25             = BM25Okapi(tokenized_corpus)
best_model       = SentenceTransformer(STAGE3_PATH)
print("✅ Reloaded from cache")
print("(Skipped — cache recovery not needed)")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Reloaded from cache
(Skipped — cache recovery not needed)


In [ ]:
# ── FULL INFERENCE PIPELINE ───────────────────────────────────────────────────
print("🚀 Running inference (Hybrid only)...")
results = []

def hybrid_retrieve(query_text, top_k=RETRIEVAL_K):
    # Dense
    q_emb = best_model.encode(
        BGE_Q_PREFIX + query_text,
        convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False
    )
    dense_scores = util.dot_score(q_emb, corpus_embeddings)[0].cpu().numpy()
    dense_ranked = np.argsort(dense_scores)[::-1][:top_k*2]

    # BM25
    bm25_scores  = bm25.get_scores(query_text.lower().split())
    bm25_ranked  = np.argsort(bm25_scores)[::-1][:top_k*2]

    # Reciprocal Rank Fusion
    K = 60
    rrf = {}
    for rank, idx in enumerate(dense_ranked):
        rrf[idx] = rrf.get(idx, 0) + 1.0/(K + rank + 1)
    for rank, idx in enumerate(bm25_ranked):
        rrf[idx] = rrf.get(idx, 0) + 1.0/(K + rank + 1)

    top_idxs = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    return [(cids[i], rrf[i]) for i in top_idxs]

for i, (qid, query_text) in enumerate(queries_val.items()):
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(queries_val)} queries done...")

    final_ranking = hybrid_retrieve(query_text)

    for rank, (cid, score) in enumerate(final_ranking):
        results.append(f"{qid}\tQ0\t{cid}\t{rank+1}\t{float(score)}\tbge_hybrid")

with open(PRED_PATH, 'w') as f:
    f.write("\n".join(results))

print(f"✅ Written {len(results)} lines → {PRED_PATH}")

🚀 Running inference (Hybrid only)...
  100/304 queries done...
  200/304 queries done...
  300/304 queries done...
✅ Written 30400 lines → /content/drive/MyDrive/TaskB/TaskB/predictions.trec


In [ ]:
pip install pytrec_eval

In [ ]:
import os
import pytrec_eval
import pandas as pd
# ── Encode corpus + build BM25 index ─────────────────────────────────────────


start_path = '/content/drive/MyDrive/TaskB/TaskB'
qrels_path = os.path.join(start_path, 'validation', 'qrels.tsv')
trec_path = os.path.join(start_path, 'predictions.trec')

qrels_df = pd.read_csv(qrels_path, sep='\t', header=None, names=['q_id', 'dummy', 'c_id', 'relevance'])
qrels = {}
for _, row in qrels_df.iterrows():
    qid = str(row['q_id'])
    if qid not in qrels:
        qrels[qid] = {}
    qrels[qid][str(row['c_id'])] = int(row['relevance'])

run = {}
with open(trec_path, 'r') as f:
    for line in f:
        qid, _, docid, rank, score, _ = line.strip().split()
        if qid not in run:
            run[qid] = {}
        run[qid][docid] = float(score)

evaluator = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut_10'})
results = evaluator.evaluate(run)

ndcg_scores = [query_measures['ndcg_cut_10'] for query_measures in results.values()]
average_ndcg = sum(ndcg_scores) / len(ndcg_scores)
print(average_ndcg)

0.3082536334360779


In [ ]:
!git clone https://github.com/TalentCLEF/talentclef26_evaluation_script.git
!pip install -r /content/talentclef26_evaluation_script/requirements.txt

Cloning into 'talentclef26_evaluation_script'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 64 (delta 7), reused 60 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 29.46 KiB | 4.21 MiB/s, done.
Resolving deltas: 100% (7/7), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 456.5/456.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.2 MB/s eta 0:00:00


In [ ]:
import os
import subprocess

start_path = '/content/drive/MyDrive/TaskB/TaskB'
old_trec_path = os.path.join(start_path, 'predictions.trec')
new_trec_path = os.path.join(start_path, 'predictions_formatted.trec')

with open(old_trec_path, 'r') as infile, open(new_trec_path, 'w') as outfile:
    for line in infile:
        outfile.write(line.replace('\t', ' '))

qrels_file = os.path.join(start_path, 'validation', 'qrels.tsv')

command = [
    "python",
    "/content/talentclef26_evaluation_script/talentclef_evaluate.py",
    "--task", "B",
    "--qrels", qrels_file,
    "--run", new_trec_path
]

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

TalentCLEF 2026 - Task B Evaluation
Received parameters:
  Task: B
  Qrels: /content/drive/MyDrive/TaskB/TaskB/validation/qrels.tsv
  Run: /content/drive/MyDrive/TaskB/TaskB/predictions_formatted.trec

Loading qrels...
Loading run...

Running Task B evaluation...

EVALUATION RESULTS

--- General Relevance (1 or 2 → relevant) ---
ndcg: 0.1861
map: 0.0594
mrr: 0.6433
precision@5: 0.3974
precision@10: 0.3868
precision@100: 0.2621

--- Graded Relevance (2 = core skill, 1 = contextual skill) ---
ndcg: 0.1920
map: 0.0594
mrr: 0.6433
precision@5: 0.3974
precision@10: 0.3868
precision@100: 0.2621



In [ ]:
!zip -j /content/drive/MyDrive/TaskB/TaskB/submission.zip /content/drive/MyDrive/TaskB/TaskB/predictions_formatted.trec

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission.zip')

In [ ]:
import shutil
import os

start_path = '/content/drive/MyDrive/TaskB/TaskB'
# This is the formatted file you generated a few steps ago
old_trec_path = os.path.join(start_path, 'predictions_formatted.trec')
# New file following the run_xx-xx_teamname.trec rule
new_trec_path = os.path.join(start_path, 'run_en-en_Olive.trec')

shutil.copy(old_trec_path, new_trec_path)

In [ ]:
!zip -j /content/drive/MyDrive/TaskB/TaskB/submission_Olive.zip /content/drive/MyDrive/TaskB/TaskB/run_en-en_Olive.trec

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission_Olive.zip')

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/TaskB/TaskB/submission_test_Olive.zip')